In [1]:
from dotenv import load_dotenv
import torch
from qdrant_client import QdrantClient
from langchain_qdrant import QdrantVectorStore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from qdrant_client.models import Distance, VectorParams
from scraper import scrape, recipe_id_to_uuid
import asyncio
import nest_asyncio
nest_asyncio.apply()


load_dotenv()

True

In [2]:
client = QdrantClient(path="./qdrant_db")
print("✅ Qdrant подключён, папка: ./qdrant_db")

✅ Qdrant подключён, папка: ./qdrant_db


In [3]:
embeddings_model = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-small",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

print("✅ Модель загружена")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Модель загружена


In [4]:
COLLECTION_NAME = "recipes"

if not client.collection_exists(COLLECTION_NAME):
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=384, distance=Distance.COSINE),
    )
    print(f'✅ Коллекция {COLLECTION_NAME} создана')
else:
    print(f'✅ Коллекция {COLLECTION_NAME} уже существует')

✅ Коллекция recipes уже существует


In [5]:
vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
    embedding=embeddings_model,
)

retriever = vector_store.as_retriever(search_kwargs={"k": 3})

print("✅ VectorStore готов")

✅ VectorStore готов


In [6]:
points, _ = client.scroll(
    collection_name=COLLECTION_NAME,
    limit=10000,
    with_payload=True,
    with_vectors=False
)

points

[Record(id='00000000-0000-0000-0000-000000002058', payload={'page_content': 'Плов с курицей. Категория: Горячие блюда, Горячие блюда в казане. Ингредиенты: Филе куриное, Морковь, Лук репчатый, Масло растительное, Рис, Приправа, Соль, Чеснок. Назначение: Для детей, На обед, На обед, На второе, На праздничный стол, 23 февраля, 8 Марта, День рождения, Новый год, Пасха, На ужин, На горячее, Неожиданные гости, Специальное питание, Для диеты. Теги: ужин, обед. Вкусы: нежный, рисовый, овощной, куриный.', 'metadata': {'id': 8280, 'title': 'Плов с курицей', 'categories': ['Горячие блюда', 'Горячие блюда в казане'], 'cuisine': '', 'ingredients': ['Филе куриное', 'Морковь', 'Лук репчатый', 'Масло растительное', 'Рис', 'Приправа', 'Соль', 'Чеснок'], 'cook_time': '90 минут', 'per100_kcal': '174.8', 'per100_protein': '8.7', 'per100_fat': '8.8', 'per100_carbs': '14.8', 'destiny': ['Для детей', 'На обед', 'На обед', 'На второе', 'На праздничный стол', '23 февраля', '8 Марта', 'День рождения', 'Новый г

In [7]:
from toolkit import RecipeToolkit

recipe_toolkit = RecipeToolkit(vector_store=vector_store)
tools = recipe_toolkit.get_tools()
tools_map = {t.name: t for t in tools}

print("Инструменты:")
for t in tools:
    print(f"  {t.name}: {t.description}")

Инструменты:
  search_recipes: Ищет рецепты в базе данных по смыслу запроса. Может исключать рецепты с определёнными ингредиентами.
  find_similar: Находит похожие рецепты по ID существующего рецепта.
  scrape_and_save_recipe: Ищет рецепты на povarenok.ru по запросу, парсит и сохраняет в базу данных.


In [8]:
tools_map

{'search_recipes': StructuredTool(name='search_recipes', description='Ищет рецепты в базе данных по смыслу запроса. Может исключать рецепты с определёнными ингредиентами.', args_schema=<class 'toolkit.SearchRecipesInput'>, func=<function RecipeToolkit.get_tools.<locals>.search_recipes at 0x1354d2e50>),
 'find_similar': StructuredTool(name='find_similar', description='Находит похожие рецепты по ID существующего рецепта.', args_schema=<class 'toolkit.FindSimilarInput'>, func=<function RecipeToolkit.get_tools.<locals>.find_similar at 0x13556c300>),
 'scrape_and_save_recipe': StructuredTool(name='scrape_and_save_recipe', description='Ищет рецепты на povarenok.ru по запросу, парсит и сохраняет в базу данных.', args_schema=<class 'toolkit.ScrapeAndSaveInput'>, func=<function RecipeToolkit.get_tools.<locals>.scrape_and_save_recipe at 0x13556c3b0>)}

In [9]:
result = tools_map["scrape_and_save_recipe"].invoke({"query": "plov", "limit": 3})
print(result)

🔍 Ищем: 'plov'
   Найдено ссылок: 3
  ✅ Иранский плов
  ✅ Плов или ПЛОВские старания
  ✅ Настоящий плов
✅ Сохранено: 3
Нашёл и сохранил 3 рецептов: Настоящий плов, Иранский плов, Плов или ПЛОВские старания
